# VoiceForge — Phase 2: PDF to Voice

This notebook demonstrates extracting text from a PDF, preparing it for narration with Groq, and synthesizing audio using Qwen3-TTS.

### Pipeline:
1. **Upload PDF** via Google Colab (`google.colab.files.upload`)
2. **Extract Text** from all pages with `pypdf`, handling empty/None pages and logging stats
3. **Narration Preparation** with **Groq** (clean headers, footers, page numbers for spoken flow)
4. **Voice Resolution** (`resolve_voice()`): custom preference or AI-selected documentary/narrative voice
5. **Speech Synthesis** with `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`
6. **Audio Playback** (`IPython.display.Audio`)

> **Prerequisite**: Set your `GROQ_API_KEY` in Google Colab Secrets (key icon in the left sidebar).

In [ ]:
# 1. Install Dependencies
!pip install -q groq pypdf qwen-tts soundfile torch torchaudio

In [ ]:
import os
import json
from pypdf import PdfReader
from groq import Groq
import soundfile as sf
from IPython.display import Audio, display

# Check Colab environment
try:
    from google.colab import files, userdata
    IN_COLAB = True
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    IN_COLAB = False
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    print("Warning: GROQ_API_KEY is not set. Please set it in Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Initialize Qwen3-TTS VoiceDesign model
try:
    import torch
    from qwen_tts import Qwen3TTSModel
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    tts_model = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        device_map=device,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    print("Qwen3-TTS VoiceDesign model loaded successfully.")
except Exception as e:
    tts_model = None
    print(f"Notice: Qwen3-TTS not loaded ({e}). Local fallback audio enabled.")

### 2. PDF Upload

In [ ]:
pdf_path = None
if IN_COLAB:
    print("Please upload a PDF file:")
    uploaded = files.upload()
    if uploaded:
        pdf_path = list(uploaded.keys())[0]
        print(f"Uploaded: {pdf_path}")
else:
    pdf_path = "sample_document.pdf"
    print(f"Running outside Colab. Using local file: {pdf_path}")

### 3. PDF Text Extraction & Statistics

In [ ]:
def extract_text_from_pdf(pdf_path, max_chars=4000):
    """
    Extract text from all pages of a PDF.
    Handles pages returning None gracefully and prints extraction statistics.
    """
    if not pdf_path or not os.path.exists(pdf_path):
        print(f"File not found: {pdf_path}")
        return ""
        
    reader = PdfReader(pdf_path)
    num_pages = len(reader.pages)
    extracted_pages = []
    empty_pages = 0
    
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            extracted_pages.append(text.strip())
        else:
            empty_pages += 1
            print(f"Warning: Page {i + 1} returned None or no extractable text.")
            
    full_text = "\n\n".join(extracted_pages)
    
    print("--- PDF Extraction Statistics ---")
    print(f"Total Pages: {num_pages}")
    print(f"Pages with Text: {len(extracted_pages)}")
    print(f"Pages with No Text: {empty_pages}")
    print(f"Total Characters Extracted: {len(full_text)}")
    print("---------------------------------\n")
    
    if len(full_text) > max_chars:
        print(f"Limiting text to {max_chars} characters for Phase 2.")
        full_text = full_text[:max_chars]
        
    return full_text

extracted_text = extract_text_from_pdf(pdf_path)
print("Sample of extracted text:")
print(extracted_text[:400] + ("..." if len(extracted_text) > 400 else ""))

### 4. Narration Preparation & Voice Resolution

In [ ]:
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "qwen/qwen3.8-27b"
]

DEFAULT_VOICE = (
    "A clear, natural, warm narrator with a neutral accent, "
    "medium pace and professional delivery."
)

def prepare_narration(text):
    """
    Cleans and prepares extracted document text into a natural spoken script using Groq.
    """
    if not text or not text.strip():
        return ""
    if not client:
        return text.strip()
        
    prompt = f"""You are an expert audio scriptwriter and narrator assistant.
Transform the following extracted text from a document into a smooth, natural spoken narration script.
- Remove all page numbers, header/footer lines, and metadata.
- Ensure natural phrasing and pronunciation-friendly flow.
- Preserve the key factual information and meaning accurately.
- Return ONLY the narration text with no conversational preamble or markdown backticks.

Document Text:
{text.strip()}
"""
    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name
            )
            narration = response.choices[0].message.content.strip()
            if narration:
                return narration
        except Exception:
            continue
    return text.strip()

def resolve_voice(text, voice_preference=""):
    """
    Resolves narrator voice instruction:
    - Preserves user preference if provided.
    - Derives structured voice instruction from text content via Groq if unspecified.
    """
    if voice_preference and voice_preference.strip():
        return voice_preference.strip()
    if not client:
        return DEFAULT_VOICE
        
    prompt = f"""You are a voice director. Based on the following narration text, determine an appropriate narrator voice.
Return ONLY a JSON object with the following fields:
- "voice_type": e.g., documentary_narrator, storyteller, news_anchor
- "gender": male, female, neutral
- "age": young, adult, mature
- "accent": neutral, British, American, Indian, etc.
- "tone": authoritative, calm, warm, energetic, etc.
- "emotion": neutral, dramatic, empathetic, etc.
- "pace": slow, medium, fast
- "style": documentary, audiobook, conversational, etc.
- "instruction": a concise, single-sentence voice description suitable for a TTS model.

Text:
{text.strip()}
"""
    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name,
                response_format={"type": "json_object"}
            )
            data = json.loads(response.choices[0].message.content)
            instruction = data.get("instruction")
            if instruction and isinstance(instruction, str) and instruction.strip():
                return instruction.strip()
        except Exception:
            continue
    return DEFAULT_VOICE

narration_script = prepare_narration(extracted_text)
voice_instruction = resolve_voice(narration_script)

print("--- Narration Script ---")
print(narration_script)
print("\n--- Voice Instruction ---")
print(voice_instruction)

### 5. Speech Synthesis & Playback

In [ ]:
def generate_audio(text, voice_instruction, output_file="output_phase2.wav"):
    print(f"Synthesizing audio for {len(text)} characters...")
    print(f"Voice: {voice_instruction}")
    
    if tts_model is not None:
        wavs, sr = tts_model.generate_voice_design(
            text=text.strip(),
            language="English",
            instruct=voice_instruction
        )
        sf.write(output_file, wavs[0], sr)
    else:
        import wave, struct, math
        sr = 24000
        total_samples = int(sr * 2.0)
        with wave.open(output_file, "w") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sr)
            frames = bytearray()
            for i in range(total_samples):
                sample = int(32767 * 0.1 * math.sin(2 * math.pi * 440 * i / sr))
                frames.extend(struct.pack("<h", sample))
            wav_file.writeframes(frames)
            
    print(f"Saved audio to: {output_file}")
    display(Audio(output_file))

generate_audio(narration_script, voice_instruction)